# Add latitude / longitude to the TTC bus delay data
Gives each delay a coordinate by matching the free-text `Location` to the TTC stop list (`stops.txt`).

**Outputs**
- `ttc_bus_delays_with_coords.csv`: delay rows with `Latitude`, `Longitude`, `Match Type`
- `ttc_bus_delays_no_location.csv`: rows that still could not be placed, with an `Unmatched Reason`

**What each kind of location becomes**

| Location text | Coordinate used |
|---|---|
| Intersection ("Dufferin and Hope") | The point where the two streets cross. Each street is rebuilt as a line through the stops on it and the lines are intersected. If that can't be done, the centre of the stops at that intersection. |
| Station ("Kipling Stn") | The station's stops |
| Single street ("Eglinton") | The middle (half the length) of that street along its stops |
| Garage / yard / division | The facility's address (looked up online), placed with the nearest TTC stop |
| Route-wide ("Entire route") | The middle of that route: among the places the route had delays, the one closest to their median |
| Loop, address, named place | That place's stops |

`Match Type` tells you which rule was used, and says "approx. name" when a misspelled or cut-off (22 character) name was matched.

In [22]:
import re
import difflib
from collections import defaultdict

import numpy as np
import pandas as pd

DELAYS_CSV = "pre_2025_ttc_bus_delays_cleaned.csv"
STOPS_TXT = "stops.txt"
OUT_MATCHED = "pre_2025_ttc_bus_delays_with_coords.csv"
OUT_UNMATCHED = "pre_2025_ttc_bus_delays_no_location.csv"

## 1. Normalise street names
Delay text and stop names are boiled down to the same key, so "Eglinton Ave West", "EGLINTON AVE. W" and "eglinton" all become `eglinton`. Note that this means Danforth Ave and Danforth Rd share a key.

In [23]:
SUFFIXES = {"ave","av","avenue","rd","road","st","street","blvd","boulevard","dr","drive","cres","crescent",
            "crt","court","ct","pkwy","parkway","lane","ln","gate","trl","trail","cir","circle","sq","square",
            "pl","place","terr","terrace","way"}
DIRS = {"w","west","e","east","n","north","s","south","wb","eb","nb","sb","side","bound","ew"}
STATION_WORDS = re.compile(r"\b(?:station|statio|stati|stat|stn|sta|subway)\b")
ALIASES = {}
# Names used in the delay data that differ from stops.txt (renamed stations, abbreviations)
STATION_ALIASES = {
    "stc": "scarboroughcentre", "stcsrt": "scarboroughcentre", "scarboroughtowncentre": "scarboroughcentre",
    "scarboroughtowncenter": "scarboroughcentre", "scartowncentre": "scarboroughcentre",
    "downsview": "sheppardwest",
}
STATION_AS_INTERSECTION = {"eglintonwest": ("eglinton", "allen")}   # Eglinton West Station is not in stops.txt
STATION_ALIAS_PREFIXES = [("scarboroughtown", "scarboroughcentre"), ("scartown", "scarboroughcentre")]
PLACE_ALIASES = {
    "yorkucommons": "yorkuniversity", "yorkuniversitycommons": "yorkuniversity", "commons": "yorkuniversity",
    "sunnybrookhospital": "sunnybrookmainentrance", "humbercollege": "humbercollegebusterminal",
    "centennialcollege": "centennialcollegebusterminal", "terminal3": "pearsonairportterminal3",
    "terminal1": "pearsonairportterminal1",
}

def core(text, strip_dirs=True):
    t = re.sub(r"\bst\.", "st ", str(text).lower())
    t = re.sub(r"[^a-z0-9 ]", " ", re.sub(r"[.'’]", "", t))
    t = re.sub(r"^(?:north|south|east|west|n|s|e|w)\s+of\s+", "", t.strip())
    toks = t.split()
    if toks and toks[0] == "the":
        toks = toks[1:]
    if len(toks) > 1 and toks[0] == "st":
        toks[0] = "saint"
    toks = ["highway" if x in ("hwy", "hiway") else x for x in toks]
    toks = [x for i, x in enumerate(toks) if i == 0 or x not in SUFFIXES]
    while strip_dirs and len(toks) > 1 and toks[-1] in DIRS:
        toks.pop()
    k = "".join(toks)
    return ALIASES.get(k, k)

def station_core(text):
    k = core(STATION_WORDS.sub(" ", str(text).lower()), strip_dirs=False)
    for pre, target in STATION_ALIAS_PREFIXES:
        if k.startswith(pre):
            return target
    return STATION_ALIASES.get(k, k)

## 2. Index the stops
`stops.txt` is one snapshot in time, so stops that have since moved or been renamed may not match older delays.

In [24]:
# ------------------------------------------------------------ index the stops
SPLIT_STOP = re.compile(r"\s+(?:at|and)\s+|\s*/\s*", re.I)
PLATFORM = re.compile(r"platform|bound\b|northbound|southbound|eastbound|westbound", re.I)

def build_index(stops):
    inter = defaultdict(list)       # frozenset({street, street}) -> [(lat, lon)]
    stations = defaultdict(list)    # "kennedy" -> coords (stops named "X Station")
    stn_tag = defaultdict(list)     # fallback: street stops tagged "- X Station"
    places = defaultdict(list)      # loops, addresses, named places
    on_street = defaultdict(list)   # street -> [(lat, lon)] of stops sitting on that street
    for name, lat, lon in zip(stops.stop_name, stops.stop_lat, stops.stop_lon):
        xy = (lat, lon)
        head, trailer = (name.split(" - ", 1) + [None])[:2] if " - " in name else (name, None)
        if trailer and not PLATFORM.search(trailer):
            if re.search(r"station\s*$", trailer, re.I):
                stn_tag[station_core(trailer)].append(xy)
            else:
                places[core(trailer)].append(xy)
        head = re.sub(r"^opposite\s+", "", head.strip(), flags=re.I)
        parts = [p for p in SPLIT_STOP.split(head) if p.strip()]
        first = re.sub(r"^\d+[a-z]?\s+", "", parts[0].strip()) if parts else ""   # "700 Arrow Rd" -> "Arrow Rd"
        if first and "station" not in first.lower() and "loop" not in first.lower():
            on_street[core(first)].append(xy)       # a stop "A at B" sits on street A
        if len(parts) == 1:
            if re.search(r"station\s*$", head, re.I):
                stations[station_core(head)].append(xy)
            else:
                places[core(head)].append(xy)
        else:
            a, b = core(parts[0]), core(parts[1])
            if a != b:
                inter[frozenset((a, b))].append(xy)
            for p in parts[:2]:
                if "loop" in p.lower():
                    places[core(p)].append(xy)
    streets = {k for pair in inter for k in pair}
    return inter, stations, stn_tag, places, streets, on_street

def centre(coords):
    a = np.array(coords)
    return float(np.median(a[:, 0])), float(np.median(a[:, 1]))

## 3. Street geometry
Streets are rebuilt as lines through their stops (in metres around Toronto) so two streets' crossing point and a street's midpoint can be computed.

In [25]:
# ------------------------------------------------------------------- geometry
# Streets are rebuilt as lines through the stops that sit on them, so two streets'
# real crossing point can be computed. Work in metres around downtown Toronto.
LAT0, LON0 = 43.70, -79.40
KX, KY = 111320 * np.cos(np.radians(LAT0)), 110540
GAP_M = 1500       # a street line is broken wherever stops are > 1.5 km apart
EXTEND_M = 800     # allow a line to be extended this far past its last stop
NEAR_CENTROID_M = 150

def to_xy(lat, lon):
    return np.array([(lon - LON0) * KX, (lat - LAT0) * KY])

def to_latlon(xy):
    return float(xy[1] / KY + LAT0), float(xy[0] / KX + LON0)

def street_lines(pts):
    """Sort a street's stops along its main direction, split where there are big gaps."""
    a = np.unique(np.round(np.array([to_xy(la, lo) for la, lo in pts]), 0), axis=0)
    if len(a) < 2:
        return []
    c = a - a.mean(0)
    axis = np.linalg.svd(c, full_matrices=False)[2][0]
    a = a[np.argsort(c @ axis)]
    gaps = np.hypot(*np.diff(a, axis=0).T)
    pieces = np.split(a, np.where(gaps > GAP_M)[0] + 1)
    return [p for p in pieces if len(p) >= 2]

def extend(line, m=EXTEND_M):
    out = line
    for end in (0, -1):
        p = line[end]
        q = line[min(2, len(line) - 1)] if end == 0 else line[max(-3, -len(line))]
        d = p - q
        n = np.hypot(*d)
        if n > 0:
            ext = (p + d / n * m)[None, :]
            out = np.vstack([ext, out]) if end == 0 else np.vstack([out, ext])
    return out

def crossings(P, Q):
    """All points where polyline P crosses polyline Q."""
    p0, p1, q0, q1 = P[:-1], P[1:], Q[:-1], Q[1:]
    d1, d2 = (p1 - p0)[:, None, :], (q1 - q0)[None, :, :]
    r = q0[None, :, :] - p0[:, None, :]
    den = d1[..., 0] * d2[..., 1] - d1[..., 1] * d2[..., 0]
    with np.errstate(divide="ignore", invalid="ignore"):
        t = (r[..., 0] * d2[..., 1] - r[..., 1] * d2[..., 0]) / den
        u = (r[..., 0] * d1[..., 1] - r[..., 1] * d1[..., 0]) / den
    ok = (np.abs(den) > 1e-9) & (t >= 0) & (t <= 1) & (u >= 0) & (u <= 1)
    i, j = np.nonzero(ok)
    return p0[i] + t[i, j, None] * d1[i, 0, :]

def cluster(points, radius=250):
    """Merge crossing points closer than `radius` m (zig-zags in the lines)."""
    out = []
    for p in points:
        for c in out:
            if np.hypot(*(c[0] / c[1] - p)) < radius:
                c[0] += p; c[1] += 1
                break
        else:
            out.append([p.copy(), 1])
    return [c[0] / c[1] for c in out]

def midpoint_of(line):
    """Point half-way along a polyline's length."""
    seg = np.hypot(*np.diff(line, axis=0).T)
    half = seg.sum() / 2
    k = int(np.searchsorted(np.cumsum(seg), half))
    before = seg[:k].sum()
    f = (half - before) / seg[k] if seg[k] else 0
    return line[k] + f * (line[k + 1] - line[k])

## 4. Matching rules

In [26]:
# ------------------------------------------------------------------- matching
class Geocoder:
    def __init__(self, stops):
        self.inter, self.stations, self.stn_tag, self.places, self.streets, on_street = build_index(stops)
        self.lines = {k: street_lines(v) for k, v in on_street.items()}
        self.lines = {k: v for k, v in self.lines.items() if v}
        self._x = {}
        self.vocab = {"street": self.streets, "station": set(self.stations) | set(self.stn_tag),
                      "place": set(self.places)}
        self.bucket = {}
        for kind, vocab in self.vocab.items():
            b = defaultdict(list)
            for k in vocab:
                b[k[:1]].append(k)
            self.bucket[kind] = b
        self._cache = {}
        self.ambiguous = False

    def resolve(self, key, kind, trunc):
        """Return [(candidate_key, exact?)]: exact, then prefix (truncated text), then fuzzy."""
        ck = (key, kind, trunc)
        if ck in self._cache:
            return self._cache[ck]
        vocab = self.vocab[kind]
        if key in vocab:
            out = [(key, True)]
        else:
            out = []
            if trunc and len(key) >= 4:
                out = [(k, False) for k in vocab if k.startswith(key)][:5]
            if not out and len(key) >= 5:
                out = [(k, False) for k in difflib.get_close_matches(key, self.bucket[kind].get(key[:1], []), n=3, cutoff=0.85)]
        self._cache[ck] = out
        return out

    def try_intersection(self, a, b, trunc):
        ka, kb = core(a), core(b)
        if not ka or not kb or ka == kb:
            return None
        for x, ex in self.resolve(ka, "street", trunc):
            for y, ey in self.resolve(kb, "street", trunc):
                if x == y:
                    continue
                pt, how = self.cross(x, y)
                if pt is not None:
                    return pt, how + ("" if ex and ey else " (approx. name)")
                if how == "ambiguous":
                    self.ambiguous = True
        return None

    def cross(self, x, y):
        """Real crossing point of streets x and y, from the street lines built from stops."""
        key = frozenset((x, y))
        if key in self._x:
            return self._x[key]
        stops_here = self.inter.get(key)
        anchor = to_xy(*centre(stops_here)) if stops_here else None
        res = (None, "no crossing")
        if x in self.lines and y in self.lines:
            for ext in (False, True):
                pts = []
                for P in self.lines[x]:
                    for Q in self.lines[y]:
                        P2, Q2 = (extend(P), extend(Q)) if ext else (P, Q)
                        pts.extend(crossings(P2, Q2))
                pts = cluster(pts)
                if anchor is not None and pts:
                    best = min(pts, key=lambda p: np.hypot(*(p - anchor)))
                    if np.hypot(*(best - anchor)) <= NEAR_CENTROID_M:
                        res = (to_latlon(best), "intersection"); break
                elif anchor is None and len(pts) == 1:
                    res = (to_latlon(pts[0]), "intersection (computed, no stop there)"); break
                elif anchor is None and len(pts) > 1:
                    res = (None, "ambiguous"); break
        if res[0] is None and anchor is not None:
            res = (to_latlon(anchor), "intersection (centre of stops)")
        self._x[key] = res
        return res

    def try_street(self, text, trunc):
        k = core(text)
        if not k:
            return None
        for x, ex in self.resolve(k, "street", trunc):
            if x in self.lines:
                longest = max(self.lines[x], key=lambda L: np.hypot(*np.diff(L, axis=0).T).sum())
                return to_latlon(midpoint_of(longest)), "street midpoint" if ex else "street midpoint (approx. name)"
        return None

    def try_station(self, text, trunc):
        k = station_core(text)
        if not k:
            return None
        if k in STATION_AS_INTERSECTION:
            pt, how = self.cross(*STATION_AS_INTERSECTION[k])
            return (pt, "station (via street crossing)") if pt else None
        for x, ex in self.resolve(k, "station", trunc):
            coords = self.stations.get(x) or self.stn_tag.get(x)
            if coords:
                return centre(coords), "station" if ex else "station (approx. name)"
        return None

    def try_place(self, text, trunc):
        k = core(text)
        k = PLACE_ALIASES.get(k, k)
        if not k:
            return None
        for x, ex in self.resolve(k, "place", trunc):
            coords = self.places.get(x)
            if coords:
                return centre(coords), "named place/address" if ex else "named place (approx. name)"
        return None

## 5. Garages and yards
Addresses come from online sources (TTC / TransitToronto facility pages). Each is placed using the nearest TTC stop: an exact address stop, the stops at the cross street, or interpolation between numbered stops. Expect these to be within a few hundred metres, not surveyed.

In [27]:
# --------------------------------------------------------- garages and yards
# Addresses looked up online (TTC / TransitToronto / Wikipedia-style facility lists), then
# placed using the TTC stop nearest that address: an exact address stop, the stops at the
# cross-street, or interpolation between numbered stops along the street.
FACILITY_SPEC = {   # key: (what it is, how it is placed, argument)
    "wilson":       ("Wilson Garage, 160 Transit Rd",                "stop", "Transit Rd at Wilson Ave North Side"),
    "arrow":        ("Arrow Road Garage, 700 Arrow Rd",              "stop", "700 Arrow Rd"),
    "birchmount":   ("Birchmount Garage, 400 Danforth Rd",           "mean", r"^(?:Danforth Rd at Birchmount Rd|Birchmount Rd at Danforth Rd)"),
    "eglinton":     ("Eglinton Garage, 38 Comstock Rd",              "mean", r"^Warden Ave at Comstock Rd"),
    "malvern":      ("Malvern Garage, 5050 Sheppard Ave E",          "interp", ("Sheppard Ave East", 5050)),
    "queensway":    ("Queensway Garage, 400 Evans Ave",              "stop", "400 Evans Ave"),
    "mcnicoll":     ("McNicoll Garage, Kennedy Rd & McNicoll Ave",   "mean", r"^(?:Kennedy Rd at Mcnicoll Ave|Mcnicoll Ave at Kennedy Rd)"),
    "mountdennis":  ("Mount Dennis Garage, 121 Industry St",         "stop", "109 Industry St"),
    "lakeshore":    ("Lakeshore Garage / Wheel-Trans, 580 Commissioners St", "stop", "600 Commissioners St"),
    "greenwood":    ("Greenwood Yard/Shops, 400 Greenwood Ave",      "mean", r"^Greenwood Ave at Danforth Ave"),
    "russell":      ("Russell Division, Queen St E at Greenwood Ave", "mean", r"^Queen St East at Greenwood Ave"),
    "roncesvalles": ("Roncesvalles Carhouse, Queen St W",            "mean", r"^Roncesvalles Ave at Queen St West"),
    "hillcrest":    ("Hillcrest Complex, 1138 Bathurst St",          "stop", "1138 Bathurst St"),
}
FACILITY_NAMES = [   # words that appear in the delay data -> facility key
    ("wilson", r"\bwilson\b"), ("eglinton", r"\begl\w*"), ("malvern", r"malvern"), ("arrow", r"\barrow\b"),
    ("birchmount", r"\bbirch|brichmount"), ("queensway", r"queens?way|queenway"), ("mcnicoll", r"mcnic"),
    ("mountdennis", r"\bm(?:ount|t)\.?\s*dennis"), ("lakeshore", r"wheel\s*-?\s*trans|lakeshore"),
    ("greenwood", r"\bgreenwood\b"), ("russell", r"\brussell\b"), ("roncesvalles", r"roncesvalles|\bronces"),
    ("hillcrest", r"hillcrest"),
]
GARAGE_WORD = re.compile(r"\b(?:garage|yard|division|carhouse|barn|shop|complex)\b", re.I)

def facility_coords(stops):
    out = {}
    for key, (label, how, arg) in FACILITY_SPEC.items():
        if how == "stop":
            m = stops[stops.stop_name == arg]
            pt = (m.stop_lat.iloc[0], m.stop_lon.iloc[0])
        elif how == "mean":
            m = stops[stops.stop_name.str.contains(arg, regex=True, case=False)]
            pt = (m.stop_lat.mean(), m.stop_lon.mean())
        else:   # interpolate along the street between the two numbered stops either side of the address
            street, num = arg
            m = stops[stops.stop_name.str.contains(rf"^(?:Opposite )?\d+ {street}$", regex=True)].copy()
            m["n"] = m.stop_name.str.extract(r"(\d+)")[0].astype(int)
            lo, hi = m[m.n <= num].sort_values("n").iloc[-1], m[m.n >= num].sort_values("n").iloc[0]
            f = (num - lo.n) / (hi.n - lo.n)
            pt = (lo.stop_lat + f * (hi.stop_lat - lo.stop_lat), lo.stop_lon + f * (hi.stop_lon - lo.stop_lon))
        out[key] = (float(pt[0]), float(pt[1]), label)
    return out

In [28]:
SPLIT_LOC = re.compile(r"\s+(?:and|at|on|@)\s+|\s*[&/@+]\s*|\s+-\s+", re.I)
HAS_STATION = re.compile(r"\b(?:station|statio|stati|stat|stn|sta)\b", re.I)
ROUTEWIDE = re.compile(r"entire|route|\brun\b|both ways|all stops|n/?a\b|unknown|^none$|various", re.I)

def geocode_one(g, raw, facilities):
    """-> (lat, lon, match_type, reason_if_unmatched)"""
    if raw is None or (isinstance(raw, float) and np.isnan(raw)) or not str(raw).strip():
        return None, None, None, "blank location"
    raw = str(raw)
    trunc = len(raw) == 22            # 2021+ locations are cut off at 22 characters
    low = raw.lower()
    has_garage_word = bool(GARAGE_WORD.search(low))

    # 1. garage / yard / division named in the text
    if has_garage_word:
        for key, pat in FACILITY_NAMES:
            if re.search(pat, low) and len(SPLIT_LOC.split(GARAGE_WORD.sub(" ", low))) == 1:
                lat, lon, _ = facilities[key]
                return lat, lon, "garage/yard (address looked up)", None

    text = GARAGE_WORD.sub(" ", raw) if has_garage_word else raw
    parts = [p.strip() for p in SPLIT_LOC.split(text) if re.search(r"[A-Za-z0-9]", p)]
    found = None
    g.ambiguous = False
    if len(parts) >= 2:
        found = g.try_intersection(parts[0], parts[1], trunc)
    if not found:
        for p in parts[:2]:
            if HAS_STATION.search(p) or station_core(p) in STATION_ALIASES.values() or core(p, False) in STATION_ALIASES:
                found = g.try_station(p, trunc)
                if found: break
    if not found:
        for p in parts[:2]:
            if len(parts) == 1 or "loop" in p.lower() or re.search(r"\d", p) or core(p) in PLACE_ALIASES:
                found = g.try_place(p, trunc)
                if found: break
    if not found and len(parts) == 1 and not ROUTEWIDE.search(raw):
        found = g.try_street(parts[0], trunc)          # a single street -> middle of the street
    if found:
        (lat, lon), kind = found
        return lat, lon, kind, None

    if has_garage_word and not parts:
        return None, None, None, "garage (which one is not specified)"
    if ROUTEWIDE.search(raw):
        return None, None, None, "route-wide"           # filled in later from the route number
    if g.ambiguous:
        return None, None, None, "ambiguous intersection (the two streets cross more than once)"
    if len(parts) == 1:
        return None, None, None, "single street or place, not found in stop list"
    return None, None, None, "no matching street/stop for this intersection"

## 6. Route midpoints

In [29]:
# ------------------------------------------------------------- route midpoints
POINT_TYPES = ("intersection", "station", "named place")

def route_midpoints(located, min_points=3):
    """`located`: rows with Route, Latitude, Longitude, Match Type for delays at real points.
    A route's 'middle' = the spot, among the distinct places it had delays, closest to the
    median of those places (so it always lies on the route)."""
    pts = located[located["Match Type"].str.startswith(POINT_TYPES)]
    pts = pts.dropna(subset=["Route"]).drop_duplicates(["Route", "Latitude", "Longitude"])
    out = {}
    for route, grp in pts.groupby("Route"):
        if len(grp) < min_points:
            continue
        xy = np.array([to_xy(a, b) for a, b in zip(grp.Latitude, grp.Longitude)])
        mid = np.median(xy, axis=0)
        best = xy[np.argmin(np.hypot(*(xy - mid).T))]
        out[route] = to_latlon(best)
    return out

## 7. Run it
Matching is done once per unique `Location` text (about 129k), then copied to every row.

In [30]:
stops = pd.read_csv(STOPS_TXT)
delays = pd.read_csv(DELAYS_CSV, low_memory=False)

facilities = facility_coords(stops)
g = Geocoder(stops)

uniq = delays["Station"].drop_duplicates()
lookup = pd.DataFrame(
    [geocode_one(g, loc, facilities) for loc in uniq],
    columns=["Latitude", "Longitude", "Match Type", "Unmatched Reason"],
)
lookup.insert(0, "Station", uniq.values)
out = delays.merge(lookup, on="Station", how="left")

# route-wide delays -> middle of that route
mids = route_midpoints(out[out["Latitude"].notna()])
route_wide = out["Unmatched Reason"] == "route-wide"
has_mid = route_wide & out["Route"].map(lambda r: r in mids)
out.loc[has_mid, "Latitude"] = out.loc[has_mid, "Route"].map(lambda r: mids[r][0])
out.loc[has_mid, "Longitude"] = out.loc[has_mid, "Route"].map(lambda r: mids[r][1])
out.loc[has_mid, "Match Type"] = "route midpoint"
out.loc[has_mid, "Unmatched Reason"] = None
out.loc[route_wide & ~has_mid, "Unmatched Reason"] = "route-wide, but route number missing or too few located delays on that route"

out[["Latitude", "Longitude"]] = out[["Latitude", "Longitude"]].round(6)
found = out["Latitude"].notna()
print(f"{found.sum():,} of {len(out):,} rows located ({found.mean():.1%})")
out.loc[found, "Match Type"].value_counts()

572,610 of 707,398 rows located (80.9%)


Match Type
station                                                  214557
intersection                                             167540
intersection (centre of stops)                            57460
route midpoint                                            30916
intersection (approx. name)                               22049
named place/address                                       17672
street midpoint                                           16459
garage/yard (address looked up)                           16264
intersection (centre of stops) (approx. name)             15365
station (approx. name)                                     5349
named place (approx. name)                                 4295
intersection (computed, no stop there)                     2064
street midpoint (approx. name)                             2057
intersection (computed, no stop there) (approx. name)       563
Name: count, dtype: int64

## 8. Save the two files

In [31]:
out.loc[found].drop(columns="Unmatched Reason").to_csv(OUT_MATCHED, index=False)

unmatched = out.loc[~found].drop(columns=["Latitude", "Longitude", "Match Type"])
unmatched.to_csv(OUT_UNMATCHED, index=False)

print(len(unmatched), "unmatched rows")
unmatched["Unmatched Reason"].value_counts()

134788 unmatched rows


Unmatched Reason
single street or place, not found in stop list                                  84342
no matching street/stop for this intersection                                   48487
garage (which one is not specified)                                              1043
blank location                                                                    793
route-wide, but route number missing or too few located delays on that route       24
ambiguous intersection (the two streets cross more than once)                      13
Name: count, dtype: int64